In [2]:
import pandas as pd
from yfetch import get_stock_history, get_stock_name
symbols = ['4GLD.DE','ESIF.DE','NUKL.DE','DFEN.DE','TDIV.AS','IWMO.MI','XDWI.DE','JEDI.DE']
rows = []
for symbol in symbols:
    history = get_stock_history(symbol, period='2y', interval='1d')
    monthly = history.resample('ME').agg({'High': 'max', 'Low': 'min'})
    monthly['Swing'] = 2 * (monthly.High - monthly.Low) / (monthly.High + monthly.Low)
    avg_swing = monthly.Swing.mean()
    last_price = history.Close.iloc[-1]
    one_year_ago = history.index[-1] - pd.DateOffset(years=1)
    past_year = history[history.index >= one_year_ago]
    prev_year = history[(history.index < one_year_ago)]
    avg_past_year = past_year.Close.mean()
    avg_prev_year = prev_year.Close.mean()
    yoy_change = avg_past_year / avg_prev_year - 1
    rows.append({
        'Symbol': symbol,
        'Name': get_stock_name(symbol),
        'Price': last_price,
        'Swing': avg_swing,
        'YoY Avg': yoy_change,
        'R/R': yoy_change / avg_swing if avg_swing > 0 else float('inf')
    })

results = pd.DataFrame(rows).set_index('Symbol')
results.to_csv('data/monthly-swing.csv')
results['Price'] = results['Price'].map('{:.2f}'.format)
for col in ['Swing', 'YoY Avg']:
    results[col] = results[col].map('{:.1%}'.format)
results

Fetched history for JEDI.DE (505 rows)


,Name,Price,Swing,YoY Avg,R/R
Symbol,,,,,
4GLD.DE,Xetra-Gold,114.45,8.8%,39.6%,4.517868
ESIF.DE,iShares MSCI Europe Financials Sector UCITS ETF,17.12,7.7%,34.0%,4.423595
NUKL.DE,VanEck Uranium and Nuclear Technologies UCITS ETF,43.08,18.8%,61.6%,3.268832
DFEN.DE,VanEck Defense ETF A USD Acc,50.68,11.0%,42.1%,3.844039
TDIV.AS,VanEck Morningstar Developed Markets Dividend ...,55.11,5.0%,23.7%,4.726806
IWMO.MI,iShares Edge MSCI World Momentum Factor UCITS ETF,97.11,7.9%,16.8%,2.129794
XDWI.DE,Xtrackers MSCI World Industrials UCITS ETF 1C,74.72,6.6%,16.9%,2.570166
JEDI.DE,VanEck Space Innovators UCITS ETF,66.63,20.1%,111.1%,5.524757
